In [50]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv


In [51]:
data = pd.read_csv("/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv",encoding="latin-1")
data = data[["v1","v2"]]
print(data.head)

<bound method NDFrame.head of         v1                                                 v2
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham              Will Ì_ b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]>


In [52]:
data["v1"] = data["v1"].map({
    "ham":0,
    "spam":1
})
print(data.head)

<bound method NDFrame.head of       v1                                                 v2
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
...   ..                                                ...
5567   1  This is the 2nd time we have tried 2 contact u...
5568   0              Will Ì_ b going to esplanade fr home?
5569   0  Pity, * was in mood for that. So...any other s...
5570   0  The guy did some bitching but I acted like i'd...
5571   0                         Rofl. Its true to its name

[5572 rows x 2 columns]>


In [53]:
# NLP Pre-Processing
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stemmer = PorterStemmer() # Converts words to their root form
stop_words = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    # OPTIONAL : text = re.sub(r'[^a-z\s]','',text) # Basically skip punctuaton marks
    # My guess is punctuation marks are more common in spams so should NOT be skipped
    words = text.split()
    words = [
        stemmer.stem(word) for word in words if word not in stop_words
    ]
    return words

data["v2"] = data["v2"].apply(preprocess)


In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    data["v2"],data["v1"],train_size=0.8
)

In [30]:
from collections import defaultdict

class MultinomialNB:
    def __init__(self):
        self.vocab = set()

        self.spam_word_count = defaultdict(int)
        self.ham_word_count = defaultdict(int)

        self.spam_messages = 0
        self.ham_messages = 0

        self.total_spam_words = 0
        self.total_ham_words = 0

    def fit(self,X,y):
        for i in range(len(y)):
            if y.iloc[i]==1:
                self.spam_messages+=1
                self.total_spam_words+=len(X.iloc[i])
            else:
                self.ham_messages+=1
                self.total_ham_words+=len(X.iloc[i])
    
            for word in X.iloc[i]:
                self.vocab.update(word)
                if y.iloc[i]==1:
                    self.spam_word_count[word]+=1
                else:
                    self.ham_word_count[word]+=1            

    def predict(self,X):
        #Priors
        log_spam = np.log(self.spam_messages/(self.spam_messages+self.ham_messages))
        log_ham = np.log(self.ham_messages/(self.spam_messages+self.ham_messages))

        for word in X:
            p_word_spam = (self.spam_word_count[word]+1)/(self.total_spam_words+len(self.vocab))
            p_word_ham = (self.ham_word_count[word]+1)/(self.total_ham_words+len(self.vocab))

            log_spam+=np.log(p_word_spam)
            log_ham+=np.log(p_word_ham)
            
        return "Spam" if (log_spam>=log_ham) else "Not Spam"

In [31]:
model = MultinomialNB()
model.fit(X_train,y_train)

In [61]:
msg1 = "This is from the bank ! Call for instant $10000"
msg2 = "Hey there !! Hope studies are going well"
msg3 = "Grades are out !!  Check your marks !!"
print(model.predict(preprocess(msg1)))
print(model.predict(preprocess(msg2)))
print(model.predict(preprocess(msg3)))

Spam
Not Spam
Not Spam


In [ ]:
# This is very basic Naive Bayes model still works quite well !!